# Training Table Walkthrough

This notebook explains and audits the pass-detection training table created from the Step 2 master join table.

Expected contract:

- the master join is split by match before feature engineering
- each output row is one non-overlapping 5-frame window, about 0.5 seconds
- train, validation, and test are written as separate Parquet files
- the target is `is_pass`, derived from `primary_event == "PASS"`
- windows where all ball position values are missing are skipped

Plain-language companion doc: `docs/training_table_walkthrough.md`.

## 1. Setup

This cell makes the notebook work whether it is opened from the project root or from the `notebooks/` folder.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits
from driblab.features.training_table import CONTINUOUS_FEATURES, OUTPUT_COLUMNS

MASTER_JOIN_PATH = MODEL_BASE_DATA_DIR / "master_join_table.parquet"
TRAINING_PATHS = {
    "train": MODEL_BASE_DATA_DIR / "training_table_train.parquet",
    "validation": MODEL_BASE_DATA_DIR / "training_table_validation.parquet",
    "test": MODEL_BASE_DATA_DIR / "training_table_test.parquet",
}
SUMMARY_PATHS = {
    split: MODEL_BASE_DATA_DIR / f"training_table_summary_{split}.csv"
    for split in TRAINING_PATHS
}

print(PROJECT_ROOT)

/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab


## 2. Rebuild the training tables

Run this from a terminal at the project root when you want to regenerate the files:

```bash
conda activate driblabvenv
PYTHONPATH=src python -m driblab.features.training_table
```

The command reads `master_join_table.parquet` and `config.yaml`, then writes one table and one summary CSV per split.

In [2]:
assert MASTER_JOIN_PATH.exists(), f"Missing {MASTER_JOIN_PATH}. Run Step 2 first."
for split, path in TRAINING_PATHS.items():
    assert path.exists(), f"Missing {path}. Run: PYTHONPATH=src python -m driblab.features.training_table"
for split, path in SUMMARY_PATHS.items():
    assert path.exists(), f"Missing {path}. Run the training table builder."

file_inventory = pd.DataFrame(
    [
        {
            "split": split,
            "table_path": str(TRAINING_PATHS[split].relative_to(PROJECT_ROOT)),
            "table_mb": TRAINING_PATHS[split].stat().st_size / 1024 / 1024,
            "summary_path": str(SUMMARY_PATHS[split].relative_to(PROJECT_ROOT)),
        }
        for split in TRAINING_PATHS
    ]
)
display(file_inventory)

,split,table_path,table_mb,summary_path
0,train,data/processed/model_base/training_table_train...,10.802245,data/processed/model_base/training_table_summa...
1,validation,data/processed/model_base/training_table_valid...,2.932422,data/processed/model_base/training_table_summa...
2,test,data/processed/model_base/training_table_test....,2.731659,data/processed/model_base/training_table_summa...


## 3. Load summaries

The summary CSVs are the quickest health check. They count windows, pass windows, no-event windows, match coverage, and period coverage.

In [3]:
summaries = pd.concat(
    [pd.read_csv(path) for path in SUMMARY_PATHS.values()],
    ignore_index=True,
)
summaries["pass_percentage"] = summaries["pass_percentage"].round(2)
display(summaries)

,split_name,total_windows,pass_windows,no_event_windows,pass_percentage,unique_matches,unique_periods
0,train,161505,15554,140892,9.63,23,2
1,validation,36719,3567,32044,9.71,5,2
2,test,35085,3725,30251,10.62,5,2


## 4. Check schemas

All split tables should have the same 25 columns, in the same order.

In [4]:
schema_checks = []
for split, path in TRAINING_PATHS.items():
    columns = pq.ParquetFile(path).schema_arrow.names
    schema_checks.append(
        {
            "split": split,
            "column_count": len(columns),
            "matches_expected_order": columns == OUTPUT_COLUMNS,
        }
    )

schema_checks = pd.DataFrame(schema_checks)
display(schema_checks)
display(pd.DataFrame({"column_order": OUTPUT_COLUMNS}))

scaler_path = PROJECT_ROOT / "artifacts/models/feature_scaler.pkl"
assert scaler_path.exists(), f"Missing scaler: {scaler_path}"

train_features = pd.read_parquet(
    TRAINING_PATHS["train"],
    columns=CONTINUOUS_FEATURES,
)
normalization_check = pd.DataFrame(
    {
        "max_abs_train_mean": [train_features.mean().abs().max()],
        "max_abs_train_std_delta": [
            (train_features.std(ddof=0) - 1).abs().max()
        ],
        "continuous_nan_count": [
            int(train_features.isna().sum().sum())
        ],
    }
)
display(normalization_check)

assert len(OUTPUT_COLUMNS) == 25
assert schema_checks["matches_expected_order"].all()
assert normalization_check.loc[0, "continuous_nan_count"] == 0

,split,column_count,matches_expected_order
0,train,25,True
1,validation,25,True
2,test,25,True


,column_order
0,t.match_id
1,t.period
2,window_time
3,data_split
4,is_attacking_direction
5,primary_event
6,is_pass
7,secondary_events
8,ball_x_avg
9,ball_y_avg


,max_abs_train_mean,max_abs_train_std_delta,continuous_nan_count
0,1.069961e-16,2.220446e-16,0


## 5. Preview rows

A few rows from each split show the modelling grain: match, period, window time, primary event, target, ball features, closest-player features, player-density features, and the team-change signal.

In [5]:
preview_columns = [
    "t.match_id",
    "t.period",
    "window_time",
    "data_split",
    "is_attacking_direction",
    "primary_event",
    "is_pass",
    "ball_x_avg",
    "ball_y_avg",
    "ball_z_avg",
    "ball_speed_avg",
    "ball_speed_change",
    "ball_direction_x",
    "ball_direction_y",
    "e.x_meters_absolute",
    "e.y_meters_absolute",
    "closest_player_team_start",
    "closest_player_team_end",
    "closest_player_dist_change",
    "n_players_near_ball",
    "n_unique_players_in_frame",
    "team_changed",
]

for split, path in TRAINING_PATHS.items():
    print(split)
    df_preview = pd.read_parquet(path, columns=preview_columns).head(5)
    display(df_preview)

train


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,...,ball_direction_x,ball_direction_y,e.x_meters_absolute,e.y_meters_absolute,closest_player_team_start,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,678949,1,3.0,train,1,no event,0,-0.036409,-0.009347,-0.338020,...,-0.001086,0.001742,-0.338840,-0.328429,unknown,1925,-0.000610,0.189225,0.066286,0
1,678949,1,3.5,train,1,no event,0,-0.058672,-0.006499,-0.332237,...,-0.064534,0.028024,-0.338840,-0.328429,1925,1925,0.038694,0.189225,0.066286,0
2,678949,1,4.0,train,1,no event,0,-0.082073,-0.003343,-0.330203,...,-0.049399,-0.021806,-0.338840,-0.328429,1925,1925,0.030603,-0.598832,0.297534,0
3,678949,1,4.5,train,1,no event,0,-0.106732,-0.009951,-0.337340,...,-0.041978,0.002693,-0.338840,-0.328429,1925,1925,0.021079,-0.598832,-0.627455,0
4,678949,1,5.0,train,1,PASS,1,-0.119583,-0.010544,-0.338804,...,-0.014911,-0.004918,0.764218,2.856264,1925,1925,0.002968,-0.598832,-0.858702,0


validation


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,...,ball_direction_x,ball_direction_y,e.x_meters_absolute,e.y_meters_absolute,closest_player_team_start,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,689526,1,0.5,validation,1,PASS,1,0.009772,-0.070614,-0.105197,...,-0.001086,0.001742,2.488091,2.260752,unknown,unknown,-0.000610,-0.598832,-3.171175,0
1,689526,1,1.0,validation,1,no event,0,-0.050011,-0.050015,0.495142,...,-0.001086,0.001742,-0.338840,-0.328429,unknown,7194,-0.000610,0.189225,1.222523,0
2,689526,1,1.5,validation,1,no event,0,-0.059140,-0.023101,1.279695,...,-0.017530,0.053712,-0.338840,-0.328429,7194,7194,-0.040121,1.765339,1.222523,0
3,689526,1,2.0,validation,1,no event,0,-0.055522,-0.008872,1.694529,...,0.162770,-0.061170,-0.338840,-0.328429,7194,7186,0.022324,0.977282,1.222523,1
4,689526,1,2.5,validation,1,no event,0,0.008797,0.021431,2.577897,...,-0.001086,0.001742,-0.338840,-0.328429,unknown,unknown,-0.000610,-0.598832,1.222523,0


test


,t.match_id,t.period,window_time,data_split,is_attacking_direction,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,...,ball_direction_x,ball_direction_y,e.x_meters_absolute,e.y_meters_absolute,closest_player_team_start,closest_player_team_end,closest_player_dist_change,n_players_near_ball,n_unique_players_in_frame,team_changed
0,713910,1,0.5,test,1,no event,0,0.069989,-0.014252,-0.297798,...,-0.001086,0.001742,-0.338840,-0.328429,unknown,unknown,-0.000610,-0.598832,-3.171175,0
1,713910,1,1.0,test,1,no event,0,0.048477,0.036813,-0.306322,...,-0.075302,0.468171,-0.338840,-0.328429,7188,7188,0.186597,0.189225,1.222523,0
2,713910,1,1.5,test,1,no event,0,0.044177,0.049393,-0.310057,...,0.013757,-0.011935,-0.338840,-0.328429,7188,7188,-0.007385,-0.598832,1.222523,0
3,713910,1,2.0,test,1,PASS,1,0.097024,-0.064275,-0.309810,...,0.075312,-0.454936,2.377231,2.260752,7188,7188,0.149554,0.189225,1.222523,0
4,713910,1,2.5,test,1,no event,0,0.101939,-0.076995,-0.301267,...,-0.001086,0.001742,-0.338840,-0.328429,unknown,7188,-0.000610,-0.598832,1.222523,0


## 6. Confirm split isolation

The training table should contain only matches assigned to that split in `config.yaml`. This is the key leakage-prevention check.

In [6]:
splits = match_splits.load_match_splits(CONFIG_PATH)
split_checks = []

for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["t.match_id", "data_split"])
    expected_matches = set(map(str, splits[split]))
    actual_matches = set(df["t.match_id"].astype(str).unique())
    split_checks.append(
        {
            "split": split,
            "all_rows_have_expected_label": bool((df["data_split"] == split).all()),
            "actual_matches": len(actual_matches),
            "expected_matches": len(expected_matches),
            "unexpected_matches": sorted(actual_matches - expected_matches),
            "missing_matches": sorted(expected_matches - actual_matches),
        }
    )

display(pd.DataFrame(split_checks))

,split,all_rows_have_expected_label,actual_matches,expected_matches,unexpected_matches,missing_matches
0,train,True,23,23,[],[]
1,validation,True,5,5,[],[]
2,test,True,5,5,[],[]


## 7. Target balance by split

The pass rate should be checked before modelling. Here it is similar across train, validation, and test, which is helpful for evaluation.

In [7]:
target_rows = []
for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["primary_event", "is_pass"])
    target_rows.append(
        {
            "split": split,
            "windows": len(df),
            "pass_windows": int(df["is_pass"].sum()),
            "pass_percentage": round(df["is_pass"].mean() * 100, 2),
            "no_event_windows": int((df["primary_event"] == "no event").sum()),
        }
    )

display(pd.DataFrame(target_rows))

,split,windows,pass_windows,pass_percentage,no_event_windows
0,train,161505,15554,9.63,140892
1,validation,36719,3567,9.71,32044
2,test,35085,3725,10.62,30251


## 8. Event distribution

`PASS` becomes the positive class. Other selected events and `no event` become negative examples.

In [8]:
event_distribution = []
for split, path in TRAINING_PATHS.items():
    events = pd.read_parquet(path, columns=["primary_event"])
    counts = events["primary_event"].value_counts().rename_axis("primary_event")
    split_counts = counts.reset_index(name="windows")
    split_counts.insert(0, "split", split)
    event_distribution.append(split_counts)

event_distribution = pd.concat(event_distribution, ignore_index=True)
display(event_distribution)

,split,primary_event,windows
0,train,no event,140892
1,train,PASS,15554
2,train,BALL TOUCH,2238
3,train,TACKLE,741
4,train,BALL RECOVERY,646
5,train,AERIAL,605
6,train,TAKEON,501
7,train,FOUL,328
8,validation,no event,32044
9,validation,PASS,3567


## 9. Audit the confirmed skip rule

The builder skips a 5-frame window only when all `t.ball_x`, `t.ball_y`, and `t.ball_z` values are missing across the whole window.

This cell estimates how many complete 5-frame windows existed in the master join after split assignment, then compares that with the generated output row counts.

In [9]:
master_cols = [
    "t.match_id",
    "t.period",
    "t.frame",
    "t.ball_x",
    "t.ball_y",
    "t.ball_z",
]
master = pd.read_parquet(MASTER_JOIN_PATH, columns=master_cols)
master = match_splits.add_data_split_column(
    master,
    splits,
    match_col="t.match_id",
)
master = master.sort_values(["t.match_id", "t.period", "t.frame"])

skip_audit_rows = []
for split, split_df in master.groupby("data_split"):
    if split not in TRAINING_PATHS:
        continue
    complete_windows = 0
    skipped_all_ball_missing = 0
    for _, period_df in split_df.groupby(["t.match_id", "t.period"], sort=False):
        complete_rows = (len(period_df) // 5) * 5
        if complete_rows == 0:
            continue
        ball = period_df.iloc[:complete_rows][["t.ball_x", "t.ball_y", "t.ball_z"]]
        ball = ball.apply(pd.to_numeric, errors="coerce").to_numpy().reshape(-1, 5, 3)
        complete_windows += len(ball)
        skipped_all_ball_missing += int(pd.isna(ball).all(axis=(1, 2)).sum())

    output_windows = int(summaries.loc[summaries["split_name"] == split, "total_windows"].iloc[0])
    skip_audit_rows.append(
        {
            "split": split,
            "complete_5_frame_windows": complete_windows,
            "skipped_all_ball_missing": skipped_all_ball_missing,
            "output_windows": output_windows,
            "complete_minus_skipped": complete_windows - skipped_all_ball_missing,
            "matches_output_count": complete_windows - skipped_all_ball_missing == output_windows,
        }
    )

display(pd.DataFrame(skip_audit_rows))

,split,complete_5_frame_windows,skipped_all_ball_missing,output_windows,complete_minus_skipped,matches_output_count
0,test,58488,23403,35085,35085,True
1,train,279437,117932,161505,161505,True
2,validation,59372,22653,36719,36719,True


## 10. Feature notes

- Windows are 5 sorted frames within one `t.match_id` and `t.period`.
- A window is skipped only when all `t.ball_x`, `t.ball_y`, and `t.ball_z` values are missing across all five frames.
- `primary_event` is selected by the smallest numeric `nearest_timestamp_distance_sec` among non-`no event` rows.
- `secondary_events` stores the other non-primary event type names from the same window.
- `is_pass` is the label and is derived only from `primary_event == "PASS"`.
- `e.x_meters_absolute` and `e.y_meters_absolute` convert only the primary event coordinates from provider 0-100 scale into pitch meters.
- Event x is flipped for period 2 and later; event y is not flipped.
- `is_attacking_direction` is period-based: `1` for period 1 and `0` for period 2 and later.
- Continuous features are normalized with `StandardScaler` after window features are built.
- The scaler is fit on the train table only, then applied to train, validation, and test.
- Missing continuous feature values are filled with `0` before scaling.
- `ball_x_avg`, `ball_y_avg`, and `ball_z_avg` start as raw tracking-coordinate averages in meters before scaling.
- `ball_speed_avg` starts as the mean valid 3D frame-to-frame ball movement before scaling.
- `ball_speed_change` starts as last valid ball movement minus first valid ball movement before scaling.
- `ball_direction_x` and `ball_direction_y` start as frame-1-to-frame-5 differences before scaling.
- Closest-player distances start as raw tracking x/y meter distances before scaling.
- `closest_player_dist_change` starts as end closest-player distance minus start closest-player distance before scaling.
- `n_players_near_ball` starts as the count of unique visible player IDs within 5 meters of the ball at least once in the window before scaling.
- `n_unique_players_in_frame` starts as the count of unique visible player IDs tracked anywhere in the window before scaling.
- `team_changed` uses event possession IDs when the primary event has a usable possession ID; otherwise it falls back to comparing closest-player teams.
- The fitted scaler is saved to `artifacts/models/feature_scaler.pkl`.

## 11. Next modelling step

The generated tables are ready to feed into a binary pass classifier. A first baseline can use the numeric columns plus encoded categorical columns for closest-player teams and primary/secondary event context, while keeping the split files separate for evaluation.